In [ ]:
import sys
if 'google.colab' in sys.modules:
    !git clone https://github.com/dmytroslav/NLP_Course.git
    %cd NLP_Course
    !pip install -r labs/lab02/requirements.txt
    !python -m spacy download uk_core_news_sm

In [ ]:
# Дані завантажуються разом з репозиторієм, додаткове монтування Google Drive не потрібне.
import pandas as pd
df = pd.read_csv('data/raw.csv')
print("Дані успішно завантажено з репозиторію.")

In [1]:
# 1. Імпорти та налаштування
import sys
import os
import pandas as pd
import json

# Додаємо папку src у шлях, щоб імпортувати наш модуль
sys.path.append(os.path.abspath('../src'))

# Імпортуємо наші функції
from preprocess import preprocess_pipeline

print("Модуль успішно імпортовано!")

# 2. Завантаження даних
df = pd.read_csv('../data/raw.csv')
# Для тесту беремо копію, щоб не псувати raw
df_processed = df.copy()

print(f"Завантажено {len(df)} записів.")

# 3. Запуск пайплайну (Processing)
# Це може зайняти хвилину для 10k текстів
print("Починаємо обробку...")

# Застосовуємо pipeline до кожного тексту
# Результат preprocess_pipeline - це словник, ми беремо з нього 'cleaned' для збереження
df_processed['processed_text'] = df_processed['Text'].apply(lambda x: preprocess_pipeline(str(x))['cleaned'])

# Також збережемо кількість речень для статистики
df_processed['num_sentences'] = df_processed['Text'].apply(lambda x: len(preprocess_pipeline(str(x))['sentences']))

print("Обробку завершено.")

# 4. Перевірка Edge Cases (Вимога 2)
print("\n--- Edge Cases Check ---")
edge_cases_path = '../tests/edge_cases.jsonl'

with open(edge_cases_path, 'r', encoding='utf-8') as f:
    edge_cases = [json.loads(line) for line in f]

for case in edge_cases[:10]: # Покажемо перші 10
    raw = case['raw_text']
    processed = preprocess_pipeline(raw)
    print(f"ID: {case['id']}")
    print(f"Raw:   {raw}")
    print(f"Clean: {processed['cleaned']}")
    print(f"Sents: {processed['sentences']}")
    print("-" * 20)

# 5. Статистика "До/Після" (Вимога 1.4)
print("\n--- Статистика ---")
print("Кількість порожніх рядків після очистки:", len(df_processed[df_processed['processed_text'] == '']))

# Дублікати
dupes_raw = df.duplicated(subset=['Text']).sum()
dupes_proc = df_processed.duplicated(subset=['processed_text']).sum()
print(f"Дублікати RAW: {dupes_raw}")
print(f"Дублікати Processed: {dupes_proc} (може зрости, бо нормалізація робить тексти схожими)")

# 6. Збереження результатів
# Зберігаємо лише потрібні колонки
output_df = df_processed[['processed_text', 'Label']].rename(columns={'processed_text': 'text', 'Label': 'label'})
output_df.to_csv('../data/processed_v2.csv', index=False)

print("\nФайл saved to data/processed_v2.csv")

Модуль успішно імпортовано!
Завантажено 10735 записів.
Починаємо обробку...
Обробку завершено.

--- Edge Cases Check ---

--- Статистика ---
Кількість порожніх рядків після очистки: 0
Дублікати RAW: 111
Дублікати Processed: 112 (може зрости, бо нормалізація робить тексти схожими)

Файл saved to data/processed_v2.csv
